# BookTrends Recommender: Exploratory Data Analysis

This notebook provides an end-to-end exploratory data analysis (EDA) workflow for the raw books and ratings datasets used in the BookTrends recommendation system. It can be re-run at any time after updating the raw data to refresh summary artifacts and figures used throughout the project.

**Quickstart:** Run the entire notebook via **Kernel → Restart & Run All** to regenerate all plots, tables, and summary exports.


In [ ]:

import os
import json
import math
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(context="talk", style="whitegrid")
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 3)

FIGS = Path("figs/eda")
FIGS.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

GENERATED_FIGS = []
GENERATED_TABLES = []

try:
    from PIL import Image
except ImportError:  # pragma: no cover - optional dependency
    Image = None


def ensure_dirs(*paths: Path) -> None:
    """Ensure that each provided directory path exists."""
    for path in paths:
        if path is None:
            continue
        Path(path).mkdir(parents=True, exist_ok=True)


def slugify(value: str, allow_unicode: bool = False) -> str:
    """Simplified slugify helper for filenames."""
    import re

    value = str(value)
    if not allow_unicode:
        value = value.encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^\w\s-]", "", value).strip().lower()
    value = re.sub(r"[-\s]+", "-", value)
    return value or "item"


def find_data_paths() -> Tuple[Optional[Path], Optional[Path]]:
    """Locate books and ratings data using preferred then legacy layout."""
    candidates = [
        {"books": Path("data/raw/books.csv"), "ratings": Path("data/raw/ratings.csv")},
        {"books": Path("datasets/books.csv"), "ratings": Path("datasets/ratings.csv")},
    ]
    for candidate in candidates:
        books_path = candidate["books"]
        ratings_path = candidate["ratings"]
        if books_path.exists():
            if ratings_path.exists():
                return books_path, ratings_path
            warnings.warn(f"Ratings file not found at {ratings_path}. Proceeding without ratings.")
            return books_path, None
    warnings.warn("Could not locate books.csv in expected directories.")
    return None, None


def assert_valid_image(path: Path, min_bytes: int = 2048, min_unique: int = 16) -> None:
    """Check that an image exists, is non-empty, and (optionally) has color variation."""
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    size = path.stat().st_size
    if size < min_bytes:
        raise AssertionError(f"Image at {path} appears too small ({size} bytes).")
    if Image is None:
        warnings.warn("Pillow not installed; skipping pixel-level validation.")
        return
    try:
        with Image.open(path) as img:
            colors = img.convert("RGB").getcolors(maxcolors=256 * 256)
            if not colors:
                raise AssertionError(f"Image at {path} has no detectable colors.")
            if len(colors) < min_unique:
                raise AssertionError(
                    f"Image at {path} may be blank (only {len(colors)} unique colors)."
                )
    except Exception as exc:  # pragma: no cover - defensive
        raise AssertionError(f"Failed to validate image {path}: {exc}")


def savefig_checked(path: Path, tight_layout: bool = True, dpi: int = 150) -> None:
    """Save a matplotlib figure and validate it."""
    ensure_dirs(path.parent)
    if tight_layout:
        try:
            plt.tight_layout()
        except Exception:
            warnings.warn("tight_layout failed; proceeding with save.")
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    assert_valid_image(path)
    GENERATED_FIGS.append(path)


books_path, ratings_path = find_data_paths()
print(f"Books path: {books_path}")
print(f"Ratings path: {ratings_path}")


## 1. Load data

We load the core books catalog and optional ratings interactions. When files are missing, the notebook will skip the relevant sections gracefully.

In [ ]:

df_books = None
if books_path and books_path.exists():
    try:
        df_books = pd.read_csv(books_path)
        print(f"Books shape: {df_books.shape}")
        print(f"Books columns: {list(df_books.columns)}")
        display(df_books.head())
    except Exception as exc:
        warnings.warn(f"Failed to read books data: {exc}")
else:
    print("[skip] books.csv not found; downstream analyses will be skipped.")


df_ratings = None
if ratings_path and Path(ratings_path).exists():
    try:
        df_ratings = pd.read_csv(ratings_path)
        print(f"Ratings shape: {df_ratings.shape}")
        print(f"Ratings columns: {list(df_ratings.columns)}")
        display(df_ratings.head())
    except Exception as exc:
        warnings.warn(f"Failed to read ratings data: {exc}")
else:
    print("[info] ratings.csv not available; ratings analyses will be skipped if needed.")

# Dtype casting for books
if df_books is not None:
    if "original_publication_year" in df_books.columns:
        years = pd.to_numeric(df_books["original_publication_year"], errors="coerce")
        years = years.where(years >= 1500)
        df_books["original_publication_year"] = years.round().astype("Int64")
    for col in ["average_rating", "ratings_count", "work_ratings_count", "work_text_reviews_count"]:
        if col in df_books.columns:
            df_books[col] = pd.to_numeric(df_books[col], errors="coerce")
    for col in ["language_code", "authors", "title"]:
        if col in df_books.columns:
            df_books[col] = df_books[col].astype(str)

if df_ratings is not None:
    for col in df_ratings.select_dtypes(include=["object"]).columns:
        df_ratings[col] = df_ratings[col].fillna("Unknown")

# Build data dictionary
column_descriptions = {
    "book_id": "Unique identifier for the book record.",
    "goodreads_book_id": "Original Goodreads identifier for the book.",
    "best_book_id": "Best edition identifier from Goodreads.",
    "work_id": "Goodreads work identifier spanning editions.",
    "books_count": "Number of editions/versions for the work.",
    "isbn": "International Standard Book Number.",
    "isbn13": "13-digit International Standard Book Number.",
    "authors": "Author(s) as provided by the source dataset.",
    "original_publication_year": "First known publication year of the work.",
    "original_title": "Original title of the work when first published.",
    "title": "Title used for this edition/record.",
    "language_code": "Language code of the edition (ISO-like).",
    "average_rating": "Average reader rating (1-5 scale).",
    "ratings_count": "Total count of ratings received across editions.",
    "work_ratings_count": "Total ratings count for the work across editions.",
    "work_text_reviews_count": "Count of text reviews for the work.",
    "ratings_1": "Count of 1-star ratings (if provided).",
    "ratings_2": "Count of 2-star ratings (if provided).",
    "ratings_3": "Count of 3-star ratings (if provided).",
    "ratings_4": "Count of 4-star ratings (if provided).",
    "ratings_5": "Count of 5-star ratings (if provided).",
    "user_id": "Identifier for the user (ratings data).",
    "rating": "Explicit rating value given by a user.",
    "timestamp": "Timestamp associated with the rating (if provided)."
}

data_dictionary: Dict[str, Dict[str, str]] = {}
if df_books is not None:
    data_dictionary["books"] = {
        col: column_descriptions.get(col, "Books dataset column.") for col in df_books.columns
    }
if df_ratings is not None:
    data_dictionary["ratings"] = {
        col: column_descriptions.get(col, "Ratings dataset column.") for col in df_ratings.columns
    }

if data_dictionary:
    ensure_dirs(PROCESSED)
    dict_path = PROCESSED / "data_dictionary.json"
    with dict_path.open("w", encoding="utf-8") as fp:
        json.dump(data_dictionary, fp, indent=2)
    print(f"Saved data dictionary to {dict_path}")
else:
    print("[skip] No datasets loaded; skipping data dictionary export.")


## 2. Data health checks

Assess missing values, duplicates, and simple validity rules to ensure the dataset is ready for downstream modeling.

In [ ]:

if df_books is None:
    print("[skip] Books data unavailable; skipping health checks.")
else:
    missing = df_books.isna().sum().sort_values(ascending=False)
    if not missing.empty:
        missing_pct = (missing / len(df_books) * 100).round(2)
        missing_df = pd.DataFrame({"missing": missing, "percent": missing_pct})
        display(missing_df.head(20))
        plt.figure(figsize=(12, 6))
        sns.barplot(x=missing_pct.index, y=missing_pct.values, color="#4c72b0")
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("% Missing")
        plt.title("Missing values by column")
        savefig_checked(FIGS / "missing_values_bar.png")
    else:
        print("[info] No missing values detected.")

    duplicate_summary = {}
    for key in ["book_id", "title", "authors"]:
        if key in df_books.columns:
            duplicate_summary[key] = int(df_books.duplicated(subset=[key]).sum())
    if duplicate_summary:
        print("Duplicate counts:")
        for k, v in duplicate_summary.items():
            print(f"  {k}: {v}")
    else:
        print("[info] No duplicate checks performed (columns missing).")

    if "average_rating" in df_books.columns:
        invalid_ratings = df_books[~df_books["average_rating"].between(1, 5, inclusive="both")].shape[0]
        if invalid_ratings:
            print(f"[warn] {invalid_ratings} records outside rating bounds [1,5].")
        else:
            print("Average ratings within expected range [1,5].")
    if "ratings_count" in df_books.columns:
        negative_counts = (df_books["ratings_count"] < 0).sum()
        if negative_counts:
            print(f"[warn] {negative_counts} records with negative ratings_count.")
        else:
            print("Ratings counts are non-negative.")


## 3. Univariate distributions (Books)

Understand the distribution of key catalog attributes such as ratings, publication year, and language coverage.

In [ ]:

if df_books is None:
    print("[skip] Books data unavailable; skipping univariate analysis.")
else:
    if "average_rating" in df_books.columns:
        plt.figure(figsize=(8, 6))
        sns.histplot(df_books["average_rating"].dropna(), bins=20, color="#dd8452")
        plt.xlabel("Average rating")
        plt.ylabel("Books")
        plt.title("Distribution of average ratings")
        savefig_checked(FIGS / "average_rating_hist.png")

        plt.figure(figsize=(8, 8))
        rating_bins = pd.cut(df_books["average_rating"], bins=np.arange(0, 5.51, 0.5))
        pie_counts = rating_bins.value_counts().sort_index()
        if not pie_counts.empty:
            pie_counts.plot.pie(autopct="%1.1f%%", colors=sns.color_palette("viridis", len(pie_counts)))
            plt.ylabel("")
            plt.title("Average rating share (0.5 bins)")
            savefig_checked(FIGS / "average_rating_pie.png")
        else:
            print("[skip] No data for average rating pie chart.")
    else:
        print("[skip] average_rating column missing.")

    if "ratings_count" in df_books.columns:
        plt.figure(figsize=(8, 6))
        sns.histplot(df_books["ratings_count"].dropna(), bins=30, color="#55a868")
        plt.xscale("log")
        plt.xlabel("Ratings count (log scale)")
        plt.ylabel("Books")
        plt.title("Distribution of ratings count")
        savefig_checked(FIGS / "ratings_count_hist_log.png")
    else:
        print("[skip] ratings_count column missing.")

    if "original_publication_year" in df_books.columns:
        year_counts = df_books.dropna(subset=["original_publication_year"])
        if not year_counts.empty:
            year_counts = year_counts[year_counts["original_publication_year"] >= 1500]
            if not year_counts.empty:
                yearly = year_counts.groupby("original_publication_year").size()
                plt.figure(figsize=(12, 6))
                yearly.plot()
                plt.xlabel("Publication year")
                plt.ylabel("Books")
                plt.title("Books published per year")
                savefig_checked(FIGS / "books_published_per_year.png")

                if "average_rating" in df_books.columns:
                    yearly_rating = (
                        df_books.groupby("original_publication_year")["average_rating"].mean().dropna()
                    )
                    if not yearly_rating.empty:
                        plt.figure(figsize=(12, 6))
                        yearly_rating.plot()
                        plt.xlabel("Publication year")
                        plt.ylabel("Mean average rating")
                        plt.title("Average rating by publication year")
                        savefig_checked(FIGS / "average_rating_by_year.png")
                    else:
                        print("[skip] No data for average rating by year plot.")
            else:
                print("[skip] No publication years >= 1500.")
        else:
            print("[skip] No publication year data after cleaning.")
    else:
        print("[skip] original_publication_year column missing.")

    if "language_code" in df_books.columns:
        lang_counts = df_books["language_code"].fillna("Unknown").value_counts()
        if not lang_counts.empty:
            plt.figure(figsize=(10, max(6, len(lang_counts.head(20)) * 0.4)))
            sns.barplot(x=lang_counts.head(20).values, y=lang_counts.head(20).index, palette="Blues_d")
            plt.xlabel("Books")
            plt.ylabel("Language code")
            plt.title("Top languages in catalog")
            savefig_checked(FIGS / "books_by_language.png")

            non_en_mask = ~df_books["language_code"].str.lower().fillna("").str.startswith("en")
            non_en_counts = df_books.loc[non_en_mask, "language_code"].value_counts()
            if not non_en_counts.empty:
                plt.figure(figsize=(10, max(6, len(non_en_counts.head(20)) * 0.4)))
                sns.barplot(x=non_en_counts.head(20).values, y=non_en_counts.head(20).index, palette="Greens_d")
                plt.xlabel("Books")
                plt.ylabel("Language code")
                plt.title("Top non-English languages")
                savefig_checked(FIGS / "books_by_language_non_en.png")
            else:
                print("[info] Non-English language subset is empty.")
        else:
            print("[skip] Language counts empty.")

        if "genres" not in df_books.columns and "primary_genre" not in df_books.columns:
            df_books["primary_genre"] = df_books["language_code"].str.upper()
            print("[info] genres column missing; using language_code as proxy primary_genre.")
    else:
        print("[skip] language_code column missing.")


## 4. Bivariate relationships

Explore how ratings relate to popularity and language groupings.

In [ ]:

if df_books is None:
    print("[skip] Books data unavailable; skipping bivariate analysis.")
else:
    if {"average_rating", "ratings_count"}.issubset(df_books.columns):
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=df_books,
            x="ratings_count",
            y="average_rating",
            alpha=0.6,
            edgecolor=None
        )
        plt.xscale("log")
        plt.xlabel("Ratings count (log scale)")
        plt.ylabel("Average rating")
        plt.title("Average rating vs ratings count")
        try:
            import statsmodels.nonparametric.smoothers_lowess as sm_lowess

            valid = df_books[["ratings_count", "average_rating"]].dropna()
            if not valid.empty:
                lowess = sm_lowess.lowess(valid["average_rating"], np.log10(valid["ratings_count"] + 1), frac=0.3)
                plt.plot(10 ** lowess[:, 0], lowess[:, 1], color="red", linewidth=2, label="LOWESS")
                plt.legend()
        except Exception:
            print("[info] statsmodels not available for LOWESS smoothing.")
        savefig_checked(FIGS / "rating_vs_count_scatter.png")
    else:
        print("[skip] average_rating or ratings_count missing; skipping scatter plot.")

    numeric_cols = df_books.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) >= 2:
        corr = df_books[numeric_cols].corr()
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
        plt.title("Correlation heatmap (numeric features)")
        savefig_checked(FIGS / "books_corr_heatmap.png")
    else:
        print("[skip] Not enough numeric columns for correlation heatmap.")

    if {"average_rating", "language_code"}.issubset(df_books.columns):
        top_langs = df_books["language_code"].value_counts().head(10).index
        subset = df_books[df_books["language_code"].isin(top_langs)]
        if not subset.empty:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=subset, x="language_code", y="average_rating", palette="Set2")
            plt.xlabel("Language code")
            plt.ylabel("Average rating")
            plt.title("Average rating by top languages")
            savefig_checked(FIGS / "rating_by_language_box.png")
        else:
            print("[skip] No data for top languages boxplot.")
    else:
        print("[skip] language_code or average_rating missing for boxplot.")


## 5. ""Hot"" and ""Cold"" books

Highlight the most popular titles and lesser-known high-quality "cold gems".

In [ ]:

if df_books is None:
    print("[skip] Books data unavailable; skipping hot/cold analysis.")
else:
    if {"average_rating", "ratings_count"}.issubset(df_books.columns):
        top10 = df_books.dropna(subset=["ratings_count"]).sort_values("ratings_count", ascending=False).head(10)
        if not top10.empty:
            top_path = PROCESSED / "top10_most_rated.csv"
            top10.to_csv(top_path, index=False)
            GENERATED_TABLES.append(top_path)
            print(f"Saved top-10 most rated books to {top_path}")

            plt.figure(figsize=(10, 6))
            sns.barplot(data=top10, x="average_rating", y="title", palette="mako")
            plt.xlabel("Average rating")
            plt.ylabel("Title")
            plt.title("Average rating of top-10 most rated books")
            savefig_checked(FIGS / "avg_rating_top10_most_rated.png")

            ratings_cols = [f"ratings_{i}" for i in range(1, 6)]
            if all(col in top10.columns for col in ratings_cols):
                ratings_totals = top10[ratings_cols].sum(axis=1)
                ratings_share = (top10[ratings_cols].div(ratings_totals, axis=0) * 100).fillna(0)
                plt.figure(figsize=(10, 6))
                bottom = np.zeros(len(top10))
                colors = sns.color_palette("Spectral", len(ratings_cols))
                for idx, col in enumerate(ratings_cols):
                    plt.barh(top10["title"], ratings_share[col], left=bottom, color=colors[idx], label=col.replace("_", " "))
                    bottom += ratings_share[col].values
                plt.xlabel("Rating share (%)")
                plt.ylabel("Title")
                plt.title("Rating distribution for top-10 most rated books")
                plt.legend(title="Rating level", bbox_to_anchor=(1.05, 1), loc="upper left")
                savefig_checked(FIGS / "ratings_dist_top10_most_rated.png")
            else:
                print("[skip] Detailed rating columns missing; skipping stacked bars for top books.")
        else:
            print("[skip] No data available for top-10 most rated analysis.")

        if not df_books["ratings_count"].dropna().empty:
            threshold = df_books["ratings_count"].dropna().quantile(0.2)
            cold = df_books[(df_books["average_rating"] >= 4.0) & (df_books["ratings_count"] <= threshold)].copy()
            cold = cold.sort_values(["average_rating", "ratings_count"], ascending=[False, True]).head(10)
            if not cold.empty:
                cold_path = PROCESSED / "top10_cold_books.csv"
                cold.to_csv(cold_path, index=False)
                GENERATED_TABLES.append(cold_path)
                print(f"Saved cold gems to {cold_path}")

                ratings_cols = [f"ratings_{i}" for i in range(1, 6)]
                if all(col in cold.columns for col in ratings_cols):
                    ratings_totals = cold[ratings_cols].sum(axis=1)
                    ratings_share = (cold[ratings_cols].div(ratings_totals, axis=0) * 100).fillna(0)
                    plt.figure(figsize=(10, 6))
                    bottom = np.zeros(len(cold))
                    colors = sns.color_palette("Spectral", len(ratings_cols))
                    for idx, col in enumerate(ratings_cols):
                        plt.barh(cold["title"], ratings_share[col], left=bottom, color=colors[idx], label=col.replace("_", " "))
                        bottom += ratings_share[col].values
                    plt.xlabel("Rating share (%)")
                    plt.ylabel("Title")
                    plt.title("Rating distribution for cold gem books")
                    plt.legend(title="Rating level", bbox_to_anchor=(1.05, 1), loc="upper left")
                    savefig_checked(FIGS / "ratings_dist_top10_cold.png")
                else:
                    print("[skip] Detailed rating columns missing; skipping stacked bars for cold books.")
            else:
                print("[info] No cold gems matched the criteria.")
    else:
        print("[skip] average_rating or ratings_count missing; skipping hot/cold analysis.")


## 6. Ratings dataset deep dive

If user-item ratings are available, inspect interaction sparsity, activity distributions, and long-tail behavior.

In [ ]:

if df_ratings is None:
    print("[skip] Ratings data unavailable; skipping ratings analysis.")
else:
    required_cols = {"user_id", "book_id", "rating"}
    if not required_cols.issubset(df_ratings.columns):
        print(f"[skip] Ratings dataset missing required columns {required_cols - set(df_ratings.columns)}.")
    else:
        n_users = df_ratings["user_id"].nunique()
        n_items = df_ratings["book_id"].nunique()
        n_interactions = len(df_ratings)
        sparsity = 1 - (n_interactions / (n_users * n_items)) if n_users and n_items else np.nan
        print(f"Unique users: {n_users}")
        print(f"Unique books: {n_items}")
        print(f"Interactions: {n_interactions}")
        print(f"Sparsity (1 - density): {sparsity:.6f}")

        user_activity = df_ratings.groupby("user_id").size()
        if not user_activity.empty:
            plt.figure(figsize=(8, 6))
            sns.histplot(user_activity, bins=30, color="#c44e52")
            plt.yscale("log")
            plt.xlabel("Interactions per user")
            plt.ylabel("Users (log scale)")
            plt.title("User activity distribution")
            savefig_checked(FIGS / "user_activity_hist.png")

        item_popularity = df_ratings.groupby("book_id").size()
        if not item_popularity.empty:
            plt.figure(figsize=(8, 6))
            sns.histplot(item_popularity, bins=30, color="#8172b2")
            plt.yscale("log")
            plt.xlabel("Interactions per book")
            plt.ylabel("Books (log scale)")
            plt.title("Item popularity distribution")
            savefig_checked(FIGS / "item_pop_hist.png")

        # Sample heatmap
        top_users = user_activity.sort_values(ascending=False).head(30).index
        top_items = item_popularity.sort_values(ascending=False).head(30).index
        subset = df_ratings[df_ratings["user_id"].isin(top_users) & df_ratings["book_id"].isin(top_items)]
        if not subset.empty:
            pivot = subset.pivot_table(index="user_id", columns="book_id", values="rating", aggfunc="mean")
            plt.figure(figsize=(10, 8))
            sns.heatmap(pivot, cmap="YlGnBu", cbar_kws={"label": "Rating"})
            plt.title("Sample user-item rating heatmap")
            savefig_checked(FIGS / "user_item_heatmap_sample.png")
        else:
            print("[skip] Not enough data for user-item heatmap sample.")

        # Long-tail coverage
        sorted_counts = item_popularity.sort_values(ascending=False)
        cum_interactions = sorted_counts.cumsum() / sorted_counts.sum()
        coverage = pd.DataFrame({
            "item_rank": np.arange(1, len(sorted_counts) + 1),
            "coverage": cum_interactions.values,
            "item_percent": np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
        })
        plt.figure(figsize=(8, 6))
        plt.plot(coverage["item_percent"] * 100, coverage["coverage"] * 100, color="#2a7f62")
        plt.xlabel("Top items (% of catalog)")
        plt.ylabel("Cumulative interactions (%)")
        plt.title("Long-tail coverage of item interactions")
        plt.grid(True, linestyle="--", alpha=0.5)
        savefig_checked(FIGS / "long_tail_coverage.png")

        # Region proxy
        df_ratings["region"] = df_ratings["user_id"].astype(int) % 3
        region_map = {0: "AMER", 1: "EMEA", 2: "APAC"}
        df_ratings["region"] = df_ratings["region"].map(region_map)
        region_counts = df_ratings["region"].value_counts(normalize=True).mul(100).round(2)
        print("Region proxy distribution (% of interactions):")
        display(region_counts.to_frame(name="percent"))


## 7. Optional: Title word cloud

Generate a visual summary of frequently occurring terms in book titles (if the `wordcloud` library is available).

In [ ]:

if df_books is None:
    print("[skip] Books data unavailable; skipping word cloud.")
else:
    try:
        from wordcloud import WordCloud, STOPWORDS

        titles = df_books["title"].dropna().astype(str)
        if titles.empty:
            print("[skip] No titles available for word cloud.")
        else:
            stopwords = STOPWORDS.union({"novel", "volume", "edition"})
            text = " ".join(titles.tolist())
            wc = WordCloud(
                width=1200,
                height=800,
                background_color="white",
                stopwords=stopwords,
                random_state=SEED,
                colormap="plasma"
            ).generate(text)
            plt.figure(figsize=(12, 8))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")
            plt.title("Common words in book titles")
            savefig_checked(FIGS / "titles_wordcloud.png")
    except ImportError:
        print("[skip] wordcloud library not installed.")


## 8. Fairness-oriented descriptive slices

Compute simple descriptive statistics across language/genre proxies and (if available) regional proxies in ratings. These are exploratory and not production fairness metrics.

In [ ]:

fairness_records = []
if df_books is None:
    print("[skip] Books data unavailable; skipping fairness slices.")
else:
    if "primary_genre" not in df_books.columns and "language_code" in df_books.columns:
        df_books["primary_genre"] = df_books["language_code"].str.upper()

    group_col = "primary_genre" if "primary_genre" in df_books.columns else "language_code"
    catalog_total = len(df_books)
    language_group = df_books.groupby(group_col)
    aggregates = {}
    if "average_rating" in df_books.columns:
        aggregates["average_rating"] = "mean"
    if "ratings_count" in df_books.columns:
        aggregates["ratings_count"] = "mean"
    if aggregates:
        lang_stats = language_group.agg(aggregates)
    else:
        lang_stats = language_group.size().to_frame(name="count")
    lang_stats = lang_stats.rename(columns={
        "average_rating": "mean_average_rating",
        "ratings_count": "mean_ratings_count"
    })
    lang_stats["share_of_catalog"] = language_group.size() / catalog_total

    if df_ratings is not None and {"book_id", "rating"}.issubset(df_ratings.columns):
        merge_cols = ["book_id", group_col]
        merged = df_books[merge_cols].merge(df_ratings, on="book_id", how="right")
        interaction_counts = merged[group_col].value_counts()
        lang_stats["share_of_interactions"] = interaction_counts / interaction_counts.sum()
    else:
        lang_stats["share_of_interactions"] = np.nan

    lang_stats = lang_stats.sort_values("share_of_catalog", ascending=False)
    display(lang_stats.head(10))

    if "mean_average_rating" in lang_stats.columns and lang_stats["mean_average_rating"].notna().any():
        plt.figure(figsize=(10, 6))
        sns.barplot(
            x=lang_stats.head(10)["mean_average_rating"],
            y=lang_stats.head(10).index,
            palette="coolwarm"
        )
        plt.xlabel("Mean average rating")
        plt.ylabel("Language/genre proxy")
        plt.title("Average rating parity snapshot by language/genre")
        savefig_checked(FIGS / "parity_language_bar.png")
    else:
        print("[skip] Unable to compute parity plot for language/genre (ratings missing).")

    if df_ratings is not None and "region" in df_ratings.columns:
        region_stats = df_ratings.groupby("region").agg({
            "rating": "mean",
            "book_id": "nunique"
        }).rename(columns={
            "rating": "mean_rating",
            "book_id": "unique_books"
        })
        region_stats["share_of_interactions"] = df_ratings.groupby("region").size() / len(df_ratings)
        display(region_stats)

        plt.figure(figsize=(8, 6))
        sns.barplot(x=region_stats.index, y=region_stats["mean_rating"], palette="Set1")
        plt.xlabel("Region proxy")
        plt.ylabel("Mean rating")
        plt.title("Average rating by region proxy")
        savefig_checked(FIGS / "parity_region_bar.png")
    else:
        print("[info] Region proxy not available; skipping region parity plot.")

    parity_snapshot = lang_stats[[col for col in ["mean_average_rating", "mean_ratings_count", "share_of_catalog", "share_of_interactions"] if col in lang_stats.columns]]
    parity_path = PROCESSED / "eda_parity_snapshot.csv"
    parity_snapshot.to_csv(parity_path)
    GENERATED_TABLES.append(parity_path)
    print(f"Saved parity snapshot to {parity_path}")


## 9. EDA summary exports

Compile key statistics into a JSON artifact for downstream consumption and print a concise recap.

In [ ]:

summary = {}
if df_books is not None:
    summary["books"] = {
        "rows": int(len(df_books)),
        "columns": list(df_books.columns),
        "missing_counts": df_books.isna().sum().to_dict(),
    }
    if "original_publication_year" in df_books.columns:
        valid_years = df_books["original_publication_year"].dropna()
        summary["books"]["min_year"] = int(valid_years.min()) if not valid_years.empty else None
        summary["books"]["max_year"] = int(valid_years.max()) if not valid_years.empty else None
    if "language_code" in df_books.columns:
        lang_counts = df_books["language_code"].value_counts()
        summary["books"]["language_count"] = int(lang_counts.size)
        summary["books"]["top_languages"] = lang_counts.head(10).to_dict()
    if (PROCESSED / "top10_most_rated.csv") in GENERATED_TABLES:
        summary["books"]["top10_most_rated_path"] = str(PROCESSED / "top10_most_rated.csv")
    if (PROCESSED / "top10_cold_books.csv") in GENERATED_TABLES:
        summary["books"]["top10_cold_books_path"] = str(PROCESSED / "top10_cold_books.csv")

if df_ratings is not None and {"user_id", "book_id"}.issubset(df_ratings.columns):
    n_users = df_ratings["user_id"].nunique()
    n_items = df_ratings["book_id"].nunique()
    n_interactions = len(df_ratings)
    sparsity = 1 - (n_interactions / (n_users * n_items)) if n_users and n_items else None
    summary["ratings"] = {
        "rows": int(n_interactions),
        "unique_users": int(n_users),
        "unique_books": int(n_items),
        "sparsity": float(sparsity) if sparsity is not None else None
    }

if summary:
    summary_path = PROCESSED / "eda_summary.json"
    with summary_path.open("w", encoding="utf-8") as fp:
        json.dump(summary, fp, indent=2)
    print(f"Saved summary to {summary_path}")
else:
    print("[skip] No data summaries to export.")

if summary:
    print("\nKey takeaways:")
    if "books" in summary:
        b = summary["books"]
        print(f"- Catalog contains {b['rows']:,} records across {len(b['columns'])} columns.")
        if b.get("language_count"):
            top_langs = ", ".join(f"{k}: {v}" for k, v in list(b["top_languages"].items())[:5])
            print(f"- Top languages by count: {top_langs}.")
        if b.get("min_year") and b.get("max_year"):
            print(f"- Publication years range from {b['min_year']} to {b['max_year']}.")
    if "ratings" in summary:
        r = summary["ratings"]
        density = 1 - r["sparsity"] if r.get("sparsity") is not None else None
        density_pct = f" ({density*100:.2f}% density)" if density is not None else ""
        print(f"- Ratings matrix has {r['rows']:,} interactions{density_pct}.")


## 10. Appendix: artifact validation

List generated artifacts and confirm they exist with reasonable size for downstream CI checks.

In [ ]:

all_paths = {
    "figures": [Path(p) for p in GENERATED_FIGS],
    "tables": [Path(p) for p in GENERATED_TABLES],
}
for category, paths in all_paths.items():
    print(f"\n{category.upper()} ({len(paths)} items)")
    for path in paths:
        exists = path.exists()
        size = path.stat().st_size if exists else 0
        print(f"- {path} :: exists={exists} size={size} bytes")
        if exists and size >= 2048 and path.suffix.lower() == ".png":
            assert_valid_image(path)
